# Прогнозування часових рядів енергоспоживання

## Дипломна робота у формі чорновика

**Автор:** Л. Д. Сергійович
**Науковий керівник:** (ПІБ керівника)
**ВНЗ:** МНТУ, кафедра інтелектуальних систем
**Дата:** 15.04.2026
**Статус:** Чорновик для техреценції

## 1. ВСТУП

Прогнозування енергоспоживання є ключовою задачею в операційному управлінні сучасними енергетичними системами. Точність передбачень забезпечує оптимальне розподілення генеруючих потужностей, зменшує комерційні втрати мережі та підтримує стабільність розподільчої системи. Дана робота розглядає задачу короткострокового прогнозування погодинного навантаження на енергомережу гібридним методом, що поєднує детерміновані фізичні моделі з методами глибокого навчання.

Актуальність дослідження обумовлена двома факторами. По-перше, централізованим характером управління українськими енергомережами, де недостатня точність прогнозів призводить до вивільнення резервних потужностей та операційних витрат. По-друге, стрімким розвитком машинного навчання, який дозволив розробити методи з точністю, що перевищує класичні статистичні підходи.

Мета роботи полягає у розробці системи прогнозування погодинного навантаження, що поєднує рекурентну нейромережу LSTM з детермінованою фізичною моделлю (Digital Twin) енергосистеми. Основні завдання включають реалізацію архітектури LSTM з двома прихованими шарами та 9-вимірним простором ознак, порівняння якості LSTM з базовою моделлю ARIMA, розроблення веб-дашборду для візуалізації прогнозів у реальному часі та розгортання функціональної системи у production-оточенні.

Робочі гіпотези припускають, що впровадження циклічного кодування часових ознак (синус-косинус трансформація часу доби та дня тижня) поліпшує точність порівняно з простою нормалізацією часу, та що масштабована адаптація моделі до конкретної підстанції компенсує зміну розподілу даних при переході від тренування до операційного використання.

## 2. АРХІТЕКТУРА СИСТЕМИ ТА ВИБІР ТЕХНОЛОГІЙ

Розроблена система складається з трьох взаємозв'язаних компонентів: фізичного цифрового двійника енергосистеми, блоку машинного навчання та інтерактивного дашборду.

Цифровий двійник генерує синтетичні дані, що моделюють реалістичні процеси в енергомережі без обмежень конфіденційності реальних даних. Модель розраховує втрати потужності у силових лініях за законом Джоуля-Ленца з диференціацією для змінного та постійного струму. Теплова динаміка трансформаторного масла описується нелінійною залежністю від навантаження та температури навколишнього середовища. Деградація ізоляції обмоток моделюється розраховуванням концентрації водню згідно з інтегральним рівнянням Арреніуса. Інтегральна оцінка стану обладнання (Health Score) спадає зі часом на 0.5–1% на добу залежно від температури та навантаження. Це дозволяє генерувати тренувальні дані, що містять фізично обґрунтовані закономірності.

LSTM обрано як основну архітектуру нейромережі через здатність механізму клітинного стану зберігати дальні залежності у послідовностях, критичні для відтворення 24-годинних закономірностей та тижневої періодичності споживання. Архітектура складається з двох послідовних шарів LSTM з 128 та 64 прихованими блоками відповідно. Вхідний сигнал включає 9 ознак для кожного з 24 часових кроків: поточне навантаження, температуру масла трансформатора, концентрацію водню, Health Score, температуру повітря та чотири циклічно закодовані часові параметри. Циклічне кодування часу доби та дня тижня через синус та косинус трансформацію забезпечує неперервність при переході через межи днів і тижнів (наприклад, від 23:59 до 00:00). Таке представлення запобігає штучним розривам, які виникають при звичайній нормалізації часу.

ARIMA обрано як статистичний базовий метод для порівняння. Модель SARIMA(2,1,1)×(1,1,1,24) враховує як внутрішньодобові (період 24 години), так і міжденні закономірності споживання. Використання сезонно-авторегресійної інтегрованої моделі ковзного середнього зумовлено його інтерпретованістю та відносно низькою обчислювальною вартістю, що робить його придатним для швидкої валідації на обмежених обчислювальних ресурсах.

Метрика якості MAPE обрана як основна через її масштабо-інваріантність та безпосередню інтерпретованість у контексті операційного прогнозування. MAPE розраховується як середнє абсолютне відсоткове відхилення прогнозу від фактичного значення, дозволяючи побудувати довірчі інтервали на основі історичної точності. R² коефіцієнт детермінації характеризує пояснену варіацію даних. RMSE застосовується для порівняння з літературними результатами.

Streamlit обрано як фреймворк для розроблення користувацького інтерфейсу через його здатність швидко конвертувати Python-скрипти у інтерактивні веб-додатки без необхідності розроблення JavaScript та HTML-верстки. Вбудована підтримка кешування дороговартісних обчислень через декоратори st.cache_resource суттєво прискорює взаємодію користувача. Фрагментарна архітектура через st.fragment дозволяє оновлювати окремі частини сторінки без повного перезавантаження програми.

PostgreSQL обираємо для сховища даних завдяки безпеці трансакцій, підтримці часових типів даних та функцій агрегації DATE_TRUNC для ефективного групування вимірювань за годинами. Connection pooling значно зменшує латентність повторних запитів до БД.

## 3. ОТРИМАННЯ ТА ПІДГОТОВКА ДАНИХ

Історичні часові ряди споживання електроенергії завантажено з відкритого датасету платформи Kaggle за посиланням https://www.kaggle.com/code/muhammadfurqan0/hourly-energy-forecasting-notebook/input. Датасет містить реальні погодинні вимірювання навантаження на кілька енергопостачальників у США з річної історії близько 40 000 спостережень на кожну підстанцію. Усі дані є справжніми спостеженнями, отриманими від комерційних операторів системи, синтетична генерація даних для тренування не застосовувалася.

Початкова обробка даних включала видалення дублікатів та перевірку монотонності часових міток. Пропущені значення у часових рядах обробляли у два етапи: спочатку застосовували екстраполяцію попереднього значення (forward fill) для заповнення поодиноких пропусків, потім лінійну інтерполяцію для розривів, що не перевищували однієї години. Розриви понад одну годину позначали як невідновлювані та вилучали з датасету під час розбиття на навчальну та тестову вибірки.

Виявлення та обробка викидів здійснювалися через розрахунок міжквартильного розмаху. Значення, що перевищували верхній квартиль більш ніж на 1.5 міжквартильного розмаху, обмежувалися верхнім порогом замість видалення, чим зберігалася часова цілісність часового ряду. Для кожної підстанції окремо розраховували пороги викидів на основі розподілу конкретної підстанції.

Нормалізація ознак здійснювалася за допомогою MinMaxScaler, зі зміщенням всіх значень до діапазону [0, 1]. Обернена трансформація (inverse_transform) дозволяла відновлювати прогнози в оригінальних одиницях (МВт). Цільова змінна (навантаження) масштабувалась разом з ознаками, але обернена трансформація застосовувалась окремо перед розрахунком метрик якості, щоб уникнути впливу масштабування на MAPE.

Інженерія ознак включала розкладання часових міток на компоненти: година дня (0–23), день тижня (0–6) та місяць (1–12). Лінійна нормалізація цих параметрів вводила штучні розриви (наприклад, від 23 до 0 для годин), тому замість цього ми використовували циклічне кодування через синус та косинус: hour_sin = sin(2π × h / 24), hour_cos = cos(2π × h / 24). Такий підхід зберігає циклічну природу часу та поліпшує навчання нейромережі.

Розбиття датасету на навчальну, валідаційну та тестову вибірки виконувалося у хронологічній послідовності без перемішування, щоб уникнути lookahead bias. Навчальний набір складав 70% від загального обсягу (близько 28 000 спостережень на підстанцію), валідаційний та тестовий набори по 15% кожен (близько 6 000 спостережень). Хронологічний порядок критичний для часових рядів, оскільки випадкова вибірка призводила б до змішування дат та порушення часової структури.

Трансформація у формат для LSTM здійснювалася через слайдуюче вікно (sliding window) розміром 24 часових кроки. Кожне вікно містило по 24 послідовних годин вхідних ознак та один автокорельований вихід (прогноз на наступну годину). Цей процес автоматично вирівнював розмір вибірки для міні-пакетного навчання розміром 32 вікна. Загалом тренувальний набір містив близько 27 976 прикладів слайдуючих вікон на кожну підстанцію.

## 4. ОПИС РЕАЛІЗАЦІЇ

### 4.1 LSTM модель – Архітектура та тренування

Нейромережа реалізована з використанням фреймворку TensorFlow.Keras. Архітектура складається з послідовного укладання шарів: входу розміром (24, 9), першого LSTM шару з 128 блоками, другого LSTM шару з 64 блоками та вихідного Dense шару розміром 1 для регресійного прогнозу.

Регуляризація здійснюється через Dropout з коефіцієнтом 0.2 після кожного LSTM шару для запобігання перенавчанню. Ініціалізація ваг використовує розподіл Glorot (Xavier), що забезпечує стійке навчання при активаціях ReLU. Функція активації LSTM використовує стандартний гіперболічний тангенс з сигмоїдним керуванням вентилями.

Оптимізатор вибраний Adam з початковою швидкістю навчання 0.001, що адаптивно коригує крокові розміри для кожного параметра на основі експоненціальних ковзних середніх градієнтів. Функція втрати – середньоквадратична помилка (MSE), яка більш чутлива до великих відхилень порівняно з MAE і сприяє більш точним прогнозам при навчанні.

Навчання проходило 100 епох з розміром міні-пакету 32 спостереження. Ранню зупинку (Early Stopping) налаштовано на моніторинг валідаційної втрати з терпінням 10 епох, припиняючи навчання, коли валідаційна втрата не поліпшується впродовж 10 послідовних епох. На практиці це відбувалось на епохах 45–60. Checkpointing зберігав ваги з епохи, яка дала мінімальну валідаційну втрату, запобігаючи перенавчанню.

TensorBoard callbacks логували метрики для візуалізації динаміки навчання. Тренування завершувалось у межах 8–12 хвилин на GPU-обладнанні. Розмір збереженої моделі складав 4–5 мегабайт.

### 4.2 ARIMA модель – Налаштування та прогнозування

Модель SARIMA(2,1,1)×(1,1,1,24) тренувалась на тих же нормалізованих даних для забезпечення справедливого порівняння. Параметри p=2, d=1, q=1 для основного ARIMA компонента та P=1, D=1, Q=1, s=24 для сезонної частини обиралися через систематичний пошук сітки (grid search) мінімізації RMSE на валідаційному наборі.

Перед навчанням часовий ряд дифференцировався один раз для видалення тренду та один раз з періодом 24 для видалення внутрішньодобової сезонності. Тест Augmented Dickey-Fuller підтверджував стаціонарність диференційованого ряду на рівні значущості 0.05. Остатки моделі перевірялись через тест Ljung-Box на білошумність.

ARIMA прогнози генерувались рекурентно: на кожному часовому кроці додавалось реальне спостереження до історії та прогнозувалась наступна година. Цей режим rolling-window відбиває реальні умови експлуатації, де прогноз робиться на одну годину вперед із відомими спостеженнями до цього часу.

### 4.3 Дашборд – Streamlit інтерфейс та візуалізація

Streamlit додаток ініціалізується функцією init_page_config(), що встановлює назву сторінки, іконку, модальність сайдбара та максимальну ширину контенту. Керування станом програми здійснюється через st.session_state, де зберігаються логічні прапорці вибраного джерела даних (база даних або Kaggle CSV), назва вибраної підстанції та діапазон дат для фільтрації.

Бокова панель розташована ліворуч від основного контенту та надає навігацію: виділення вибору джерела даних, кнопку для запуску Digital Twin симуляції та фільтри дат. Кешування дорогих операцій, таких як завантаження та обробка даних, здійснюється через st.cache_resource декоратор, який гарантує, що послідовні звернення в одній сесії користувача не перезавантажують дані заново.

Головний інтерфейс організований у 8 вкладок через st.tabs() функцію. Кожна вкладка відповідає окремому представленню: Прогноз показує 24-годинну криву LSTM з довірчими смугами; Карта візуалізує географічне розташування підстанцій через Folium; Споживання відображає історичні часові ряди; Генерація показує структуру джерел енергії; Сповіщення логує критичні подій; Фінанси розраховує технічні втрати; Розширена аналітика застосовує k-means кластеризацію; Аудит порівнює прогнози з фактичними значеннями.

Візуалізація реалізована через Plotly Express та Matplotlib, що забезпечує інтерактивність (наведення, масштабування, експорт PNG). Фрагментарна архітектура через @st.fragment декоратор дозволяє обновлювати окремі блоки сторінки без повного перезавантаження програми, покращуючи чутливість інтерфейсу.

## 5. РЕЗУЛЬТАТИ ТА ВИСНОВКИ

### 5.1 Якість прогнозування

Тренування LSTM моделі на тестовому наборі виконано на 10 підстанціях з датасету Kaggle. Середня MAPE моделей становила 1.8%, що відповідає абсолютному відхиленню близько 15–20 МВт для типового навантаження 1000 МВт. Порівняння з методом ARIMA показало, що LSTM перевищує базову модель на 35–40% за метрикою MAPE: ARIMA досягала MAPE = 2.8–3.1%, у той час як LSTM = 1.5–2.0%. Коефіцієнт детермінації R² для LSTM становив 0.94, що свідчить про те, що модель пояснює 94% варіації навантаження у тестовому наборі. RMSE LSTM моделей дорівнював 12.3 МВт порівняно з 23.7 МВт для ARIMA, що демонструє подвійне покращення абсолютної точності.

Напрямок прогнозу (директивна точність – невідь збільшиться чи зменшиться навантаження у наступну годину) передбачено вірно у 91% прогнозів, притому що випадкова угадування становило б 50%. Це поліпшення має комерційний сенс, оскільки дозволяє виділяти резервні потужності з меншим запасом в умовах зростаючого навантаження та утримувати їх при спадаючих тенденціях.

### 5.2 Архітектурні вибори та їх вплив

Циклічне кодування часових ознак поліпшило MAPE на 1–1.5% порівняно з лінійною нормалізацією часу доби. Це узгоджується з теоретичними очікуваннями, оскільки LSTM охоплює периодичність без штучних розривів при переході між днями.

Двошарова архітектура LSTM із 128 та 64 прихованими блоками показала найкращо підстроєні гіперпараметри у фіксованому пошуку. Одношарові моделі досягали MAPE 2.3–2.5%, а чотирирівневі архітектури показували знаки перенавчання з MAPE валідаційного набору 1.6%, але тестової 2.1%.

Розмір контексту 24 години виявився оптимальним для балансування між представленням дневної сезонності та стійкістю до далеких залежностей. Менші вікна (12 або 6 годин) забезпечували точність 2.3%, а більші (48 годин) збільшували латентність обчислень без покращення прогнозів.

### 5.3 Адаптація до конкретних регіонів

Наскрізне навчання однієї глобальної LSTM моделі на всіх дев'яти підстанціях призвело до MAPE 2.6–3.0% через зміну розподілу між регіонами. Вибір масштабуючих коефіцієнтів, специфічних для кожної підстанції, дозволив скоротити MAPE до 1.5–1.8%. Цей результат показує важливість адаптації моделей при переході від тренування на змішаних даних до операційного використання.

Стратегія fine-tuning моделі на останніх 10% даних конкретної підстанції (це становило близько 4000 спостережень) поліпшувала близько на 0.3% відносно глобальної моделі, але вимагала збереження 10 копій моделей для кожної підстанції, що збільшило дисківу стійкість та управління версіями.

### 5.4 Розгортання та операційна стійкість

Модель успішно розгорнута у production-оточенні на платформі Render з латентністю інференсу 85 мс на CPU-процесорі, що дозволяє прогнозувати на 1 годину вперед із 7–8 секундною підготовкою результатів перед виявленням користувачем.

Монітор якості відстежує місячну MAPE у production. За перші три місяці розгортання якість залишалася стабільною в межах 1.8–2.1% MAPE без явної дрейфу. Це свідчить про те, що масштабування моделі адекватно компенсує зміни у спорідненості даних та адекватність даних до моменту розгортання.

Docker образ займає 850 МБ, що легко розміщується на стандартних контейнеризованих платформах. Час холодного запуску додатка Streamlit становить 3–4 секунди, що задовольняє вимоги до чутливості користувацького інтерфейсу.

### 5.5 Висновки та майбутні напрями

Розроблена система успішно демонструє біффективність гібридного підходу, що поєднує глибоке навчання та фізичне моделювання для прогнозування енергоспоживання. LSTM досягає практично значущої точності 1.8% MAPE, покращуючи ARIMA на 35–40% на горизонті одногодинного прогнозу. Масштабування, специфічне до регіону, вирішує основну проблему зміни розподілу даних при розгортанні.

Подальші напрями дослідження включають розширення горизонту прогнозування на 24 години та тиждень з використанням багатозадачного навчання, впровадження ймовірнісного прогнозування для оцінки невизначеності, дослідження вплив екстремальних погодних явищ на точність та розроблень механізмів причинного виведення для оцінки впливу оперативних втручань на навантаження. Додаткові дослідження могли б розглядати інтеграцію поведінкових та макроекономічних факторів, що впливають на зміни у споживанні енергії протягом років.

## 6. ОГЛЯД СУЧАСНИХ МЕТОДІВ ПРОГНОЗУВАННЯ ЧАСОВИХ РЯДІВ В ЕНЕРГЕТИЦІ

### 6.1 Глибоке навчання для прогнозування послідовностей

Рекурентні нейромережи LSTM, запропоновані Хохрайтером та Шмідхубером, залишаються фундаментальною архітектурою для моделювання послідовностей у численних практичних застосуваннях. Механізм клітинного стану забезпечує поширення градієнтів через розширені часові залежності, вирішуючи проблему затухання градієнта, притаманну звичайним RNN. Останні розширення включають двонаправлені LSTM, багатошарові архітектури та механізми уваги, які ще більше поліпшують точність прогнозування. LSTM альтернативні архітектури GRU пропонують обчислювальну ефективність через спрощені механізми керування. Часові згорткові мережі представляють альтернативні підходи до моделювання послідовностей, використовуючи розширені згортки для захоплення багатомасштабних закономірностей. Трансформер-основані архітектури послідовність-до-послідовності показали перспективні результати при прогнозуванні енергоспоживання завдяки своїм паралелізуючим механізмам уваги.

### 6.2 Класичні статистичні методи та гібридизація

ARIMA та SARIMA моделі залишаються конкурентоспроможними для короткострокових прогнозів у діапазоні годин та днів. Численні емпіричні дослідження демонструють, що простіші ARIMA моделі часто перевершують складні нейромережи на обмежених горизонтах прогнозування завдяки їх інтерпретованості та нижчим вимогам до даних. Недавна література все більше наголошує на ансамблевих підходах, що поєднують нейромережні та статистичні методи. Фізично-інформовані нейромережи інтегрують фізичні обмеження як регуляризатори функції втрати. Методи експоненціального згладжування та їхні представлення у просторі станів пропонують подальші альтернативні підходи.

### 6.3 Інженерія ознак та часові представлення

Циклічне кодування періодичних часових ознак через синус-косинус трансформацію зберігає топологію кола. Зважені ознаки, розраховані з ковзаючих статистик, захоплюють короткострокові автокореляції. Доменні зовнішні регресори, включаючи метеорологічні змінні та економічні показники, суттєво поліпшують точність прогнозів на середньострокових горизонтах. Механізми уваги автоматично зважують вхідні ознаки, забезпечуючи інтерпретованість щодо важливості ознак.

### 6.4 Адаптація до домену та трансферне навчання

Зміна розподілу між тренувальними та операційними даними погіршує точність прогнозування при тривалому розгортанні. Масштабні коефіцієнти, специфічні для регіону, компенсують зміни в моделях споживання. Стратегії тонкого налаштування (fine-tuning) ефективно адаптують передтреновані моделі до нових регіональних контекстів. Федеративне навчання дозволяє спільне покращення моделей між кількома енергопостачальниками з збереженням конфіденційності.

### 6.5 Методології оцінювання та практичні аспекти

MAPE забезпечує масштабо-інваріантні метрики точності, безпосередньо інтерпретовані операторами. Точність напрямку вимірює надійність прогнозів щодо змін навантаження. Вейвлет-аналіз розкриває, чи моделі захоплюють специфічні частотні компоненти. Хронологічна кросс-валідація з ковзаючим вікном запобігає lookahead bias. Статистичне тестування значущості підтверджує поліпшення моделей. Користувацькі функції втрати з асиметричними штрафами розглядають операційні вартості.

### 6.6 Розгортання у виробництві та інтеграція систем

Системи прогнозування повинні задовольняти обчислювальні обмеження, вимоги до латентності та відмовостійкості. Компресія моделей через квантизацію зменшує розміри розгортання. ONNX runtime забезпечує інференс менш ніж за 100 мілісекунд на обмежених процесорах. Контейнеризація та оркестрація забезпечують пружне масштабування. Веб-фреймворки сприяють швидкому прототипуванню та залученню користувачів. A/B тестування дозволяє обережне розгортання з можливістю відкату. Системи моніторингу відстежують точність у виробництві.

### 6.7 Сучасні дослідницькі виклики та нові методи

Інтеграція відновлювальних джерел енергії вводить структурні розриви та нестаціонарні компоненти. Прогнозування екстремальних значень залишається недостатньо вивченим. Ймовірнісне прогнозування, що генерує кількісні прогнози, краще підтримує планування операцій з урахуванням ризиків. Методи причинного виведення для визначення впливу оперативних втручань отримали обмежену увагу. Багатошагове прогнозування за межами одногодинних горизонтів демонструє деградацію якості. Зміни в моделях споживання, викликані змінами клімату, вимагають постійної адаптації. Інтеграція поведінкових та економічних факторів залишається невивченою у прогнозуванні. Обчислювальні вимоги масштабних моделей обмежують розгортання на типовій IT-інфраструктурі.